# Knowledge Loom to FORRT Nanopublications

This notebook converts a [TIB Knowledge Loom](https://knowledgeloom.tib.eu) record into [FORRT Knowledge Loom](https://w3id.org/np/RALIq4JelUP-q9BuWONcKMJ87B5n59ppcwhQjl-1dheO4) nanopublications.

For each loom record, we generate:
1. **FORRT Claims** — one per Knowledge Loom statement (the AIDA sentence expressing the scientific finding)
2. **One FORRT-KL Replication Study** — aggregates computational metadata (methods, packages, runtime, input data, scripts)
3. **FORRT-KL Replication Outcomes** — one per analysis step, with machine-readable proof URL, analysis type, and key result

Data sources:
- **Knowledge Loom API** (`/api/v1/articles/`) — statements (claims), concepts, authors, datasets
- **GitLab repos** (`loom-records/`) — dtreg JSON-LD files with computational provenance

## Setup

In [ ]:
# Install dependencies (uncomment if needed)
# !pip install nanopub rdflib

In [1]:
import json
import re
import urllib.parse
import urllib.request
from dataclasses import dataclass, field
from datetime import datetime, timezone
from pathlib import Path

from rdflib import Dataset, Namespace, URIRef, Literal
from rdflib.namespace import RDF, RDFS, XSD, FOAF
from nanopub import Nanopub, NanopubConf, load_profile

## Configuration

In [ ]:
# Which loom record to convert
# Use the Knowledge Loom resource ID (from the URL: knowledgeloom.tib.eu/resource/<ID>)
LOOM_RESOURCE_ID = "a03gjbxh5v"  # Iris dataset analysis

# Optional: GitLab repo name (for dtreg JSON-LD files). Leave empty to skip dtreg parsing.
LOOM_GITLAB_REPO = ""  # e.g. "mills-2020-1-1"

# Profile for signing
PROFILE_PATH = "/Users/annef/Documents/ScienceLive/annefou-profile/profile.yml"

# Output directory
OUTPUT_DIR = "../outputs/"

# Set to True to publish to the nanopub network
PUBLISH = False

In [3]:
# Load nanopub profile
profile = load_profile(PROFILE_PATH)
print(f"Profile: {profile.name} ({profile.orcid_id})")

Profile: Anne Fouilloux (https://orcid.org/0000-0002-1784-2920)


## Namespaces and constants

In [ ]:
PUBLISHER = "https://sciencelive4all.org/"

# Use sciencelive namespace for nanopub URIs
TEMP_NP = Namespace("https://w3id.org/sciencelive/np")

# Standard namespaces
NP = Namespace("http://www.nanopub.org/nschema#")
DCT = Namespace("http://purl.org/dc/terms/")
NT = Namespace("https://w3id.org/np/o/ntemplate/")
NPX = Namespace("http://purl.org/nanopub/x/")
PROV = Namespace("http://www.w3.org/ns/prov#")
ORCID = Namespace("https://orcid.org/")
SCHEMA = Namespace("http://schema.org/")
SCIENCELIVE = Namespace("https://w3id.org/sciencelive/o/terms/")
HYCL = Namespace("http://purl.org/petapico/o/hycl#")

# Template URIs
FORRT_CLAIM_TEMPLATE = URIRef("https://w3id.org/np/RAVdxfm3fgFahBItmNmJX_Xkmg1xlimDtoSMjZgNIs2bQ")
FORRT_KL_STUDY_TEMPLATE = URIRef("https://w3id.org/np/RALIq4JelUP-q9BuWONcKMJ87B5n59ppcwhQjl-1dheO4")
FORRT_KL_OUTCOME_TEMPLATE = URIRef("https://w3id.org/np/RAw3XdUhxQJfKBaU-cQhV6c7au4rLd5CSUdbMKTS_FB8g")

# Provenance/pubinfo templates (same as used by Nanodash)
PROV_TEMPLATE = URIRef("https://w3id.org/np/RA7lSq6MuK_TIC6JMSHvLtee3lpLoZDOqLJCLXevnrPoU")
PUBINFO_TEMPLATE_1 = URIRef("https://w3id.org/np/RACJ58Gvyn91LqCKIO9zu1eijDQIeEff28iyDrJgjSJF8")
PUBINFO_TEMPLATE_2 = URIRef("https://w3id.org/np/RAukAcWHRDlkqxk7H2XNSegc1WnHI569INvNr-xdptDGI")

# Knowledge Loom API
KL_API = "https://knowledgeloom.tib.eu/api/v1"

# GitLab API
GITLAB_API = "https://gitlab.com/api/v4"
LOOM_GROUP = "TIBHannover/lki/knowledge-loom/loom-records"

# dtreg analysis type mapping (ePIC hash -> sciencelive URI)
DTREG_TYPE_MAP = {
    "feeb33ad3e4440682a4d": SCIENCELIVE["dtreg-DataAnalysis"],
    "37182ecfb4474942e255": SCIENCELIVE["dtreg-DataPreprocessing"],
    "5b66cb584b974b186f37": SCIENCELIVE["dtreg-DescriptiveStatistics"],
    "b9335ce2c99ed87735a6": SCIENCELIVE["dtreg-GroupComparison"],
    "286991b26f02d58ee490": SCIENCELIVE["dtreg-RegressionAnalysis"],
    "3f64a93eef69d721518f": SCIENCELIVE["dtreg-CorrelationAnalysis"],
    "c6b413ba96ba477b5dca": SCIENCELIVE["dtreg-MultilevelAnalysis"],
    "6e3e29ce3ba5a0b9abfe": SCIENCELIVE["dtreg-ClassPrediction"],
    "c6e19df3b52ab8d855a9": SCIENCELIVE["dtreg-ClassDiscovery"],
    "5e782e67e70d0b2a022a": SCIENCELIVE["dtreg-AlgorithmEvaluation"],
    "437807f8d1a81b5138a3": SCIENCELIVE["dtreg-FactorAnalysis"],
}
ANALYSIS_TYPE_HASHES = set(DTREG_TYPE_MAP.keys())

print("Namespaces configured.")

## Helper functions

In [ ]:
def fetch_json(url):
    req = urllib.request.Request(url)
    with urllib.request.urlopen(req) as resp:
        return json.loads(resp.read().decode())

def fetch_text(url):
    req = urllib.request.Request(url)
    with urllib.request.urlopen(req) as resp:
        return resp.read().decode()

def gitlab_api_url(repo_path, endpoint):
    encoded = urllib.parse.quote(f"{LOOM_GROUP}/{repo_path}", safe="")
    return f"{GITLAB_API}/projects/{encoded}/{endpoint}"

def slugify(s):
    return re.sub(r'[^a-zA-Z0-9_-]', '-', s.lower()).strip('-')[:60]

print("Helper functions defined.")

## Fetch data from Knowledge Loom API

In [ ]:
# Fetch the article from the Knowledge Loom API
kl_data = fetch_json(f"{KL_API}/articles/get_article_by_id/?id={LOOM_RESOURCE_ID}")

article = kl_data.get("article", kl_data)
statements = kl_data.get("statements", [])
datasets = kl_data.get("basises", [])

print(f"Title: {article['name']}")
print(f"Authors: {', '.join(a['name'] for a in article.get('authors', []))}")
print(f"DOI: {article.get('dois', 'none')}")
print(f"Research fields: {', '.join(f['label'] for f in article.get('research_fields', []))}")
print(f"Concepts: {', '.join(c['label'] for c in article.get('concepts', []))}")
print(f"\nStatements ({len(statements)}):")
for s in statements:
    doi_str = f" [DOI: {s['doi']}]" if s.get('doi') else ""
    print(f"  - {s['label']}{doi_str}")
    print(f"    Type: {s.get('type', {}).get('name', '?')} | Concepts: {', '.join(c['label'] for c in s.get('concepts', []))}")
print(f"\nDatasets ({len(datasets)}):")
for d in datasets:
    print(f"  - {d['name']} ({d.get('id', 'no DOI')})")

In [ ]:
# Optional: fetch dtreg JSON-LD files from GitLab for computational metadata
@dataclass
class AnalysisInfo:
    label: str = ""
    method_label: str = ""
    method_call: str = ""
    package_name: str = ""
    package_version: str = ""
    runtime_name: str = ""
    runtime_version: str = ""
    input_label: str = ""
    input_source_url: str = ""
    input_rows: int = 0
    input_cols: int = 0
    output_rows: int = 0
    output_cols: int = 0
    analysis_type_hash: str = ""
    json_file: str = ""

def get_prop(obj, suffix):
    if not isinstance(obj, dict): return None
    for key, val in obj.items():
        if key.endswith(f"#{suffix}") or key == suffix: return val
    return None

def extract_analysis_step(step, filename):
    info = AnalysisInfo(json_file=filename)
    step_type = step.get("@type", "")
    if step_type.startswith("doi:"):
        h = step_type.split("doi:")[-1]
        if h in ANALYSIS_TYPE_HASHES: info.analysis_type_hash = h
    executes = get_prop(step, "executes")
    if isinstance(executes, dict):
        info.method_label = get_prop(executes, "label") or ""
        info.method_call = get_prop(executes, "is_implemented_by") or ""
        part_of = get_prop(executes, "part_of")
        if isinstance(part_of, dict):
            info.package_name = get_prop(part_of, "label") or ""
            info.package_version = get_prop(part_of, "version_info") or ""
            runtime = get_prop(part_of, "part_of")
            if isinstance(runtime, dict):
                info.runtime_name = get_prop(runtime, "label") or ""
                info.runtime_version = get_prop(runtime, "version_info") or ""
    has_input = get_prop(step, "has_input")
    if isinstance(has_input, dict):
        info.input_label = get_prop(has_input, "label") or ""
        info.input_source_url = get_prop(has_input, "source_url") or ""
        chars = get_prop(has_input, "has_characteristic")
        if isinstance(chars, dict):
            info.input_rows = get_prop(chars, "number_of_rows") or 0
            info.input_cols = get_prop(chars, "number_of_columns") or 0
    has_output = get_prop(step, "has_output")
    if isinstance(has_output, dict):
        chars = get_prop(has_output, "has_characteristic")
        if isinstance(chars, dict):
            info.output_rows = get_prop(chars, "number_of_rows") or 0
            info.output_cols = get_prop(chars, "number_of_columns") or 0
    method_str = f"{info.package_name}::{info.method_label}" if info.package_name else info.method_label
    info.label = method_str or filename
    return info if (info.method_label or info.analysis_type_hash) else None

analyses = []
gitlab_url = ""
script_url = ""
paper_doi = article.get("dois", "")
data_doi = ""

if LOOM_GITLAB_REPO:
    gitlab_url = f"https://gitlab.com/{LOOM_GROUP}/{LOOM_GITLAB_REPO}"
    tree = fetch_json(gitlab_api_url(LOOM_GITLAB_REPO, "repository/tree?per_page=100&ref=main"))
    json_files = [f["name"] for f in tree if f["name"].endswith(".json")]
    r_scripts = [f["name"] for f in tree if f["name"].endswith(".R")]
    if r_scripts:
        script_url = f"{gitlab_url}/-/blob/main/{r_scripts[0]}"
        content = fetch_text(gitlab_api_url(LOOM_GITLAB_REPO, f"repository/files/{urllib.parse.quote(r_scripts[0], safe='')}/raw?ref=main"))
        for line in content.split("\n"):
            ll = line.lower().strip()
            if ll.startswith("#"):
                if "doi paper" in ll or "doi article" in ll:
                    m = re.search(r'https://doi\.org/[^\s"]+', line)
                    if m: paper_doi = m.group(0)
                elif "doi data" in ll:
                    m = re.search(r'https://doi\.org/[^\s"]+', line)
                    if m: data_doi = m.group(0)
    for jf in json_files:
        try:
            data = fetch_json(gitlab_api_url(LOOM_GITLAB_REPO, f"repository/files/{urllib.parse.quote(jf, safe='')}/raw?ref=main"))
        except: continue
        has_part = get_prop(data, "has_part")
        steps = [has_part] if isinstance(has_part, dict) else (has_part if isinstance(has_part, list) else [data])
        for step in steps:
            if isinstance(step, dict):
                info = extract_analysis_step(step, jf)
                if info: analyses.append(info)
    print(f"GitLab: {len(analyses)} analysis steps from {len(json_files)} JSON-LD files")
    if paper_doi: print(f"Paper DOI: {paper_doi}")
else:
    print("GitLab repo not specified — skipping dtreg JSON-LD parsing.")
    print("FORRT Claims will still be generated from Knowledge Loom statements.")

## Create FORRT Claim nanopubs (from Knowledge Loom statements)

In [ ]:
def create_forrt_claim(statement, article, resource_id, profile):
    """Create a FORRT Claim nanopub from a Knowledge Loom statement."""
    this_np = URIRef(TEMP_NP)
    head_graph = URIRef(TEMP_NP + "/Head")
    assertion_graph = URIRef(TEMP_NP + "/assertion")
    provenance_graph = URIRef(TEMP_NP + "/provenance")
    pubinfo_graph = URIRef(TEMP_NP + "/pubinfo")
    
    author_uri = ORCID[profile.orcid_id]
    claim_id = slugify(f"kl-claim-{statement['statement_id']}")
    claim_uri = URIRef(TEMP_NP + f"/{claim_id}")
    
    # The statement label IS the AIDA sentence
    aida_sentence = statement["label"]
    aida_uri = URIRef(f"http://purl.org/aida/{urllib.parse.quote(aida_sentence, safe='')}")
    
    # Determine FORRT claim type from the analysis type
    analysis_type = statement.get("type", {})
    type_hash = analysis_type.get("type_id", "")
    # Default to computational_performance for now
    forrt_type = SCIENCELIVE["computational_performance-FORRT-Claim"]
    
    ds = Dataset()
    ds.bind("this", TEMP_NP)
    ds.bind("sub", TEMP_NP)
    ds.bind("np", NP)
    ds.bind("dct", DCT)
    ds.bind("nt", NT)
    ds.bind("npx", NPX)
    ds.bind("xsd", XSD)
    ds.bind("rdfs", RDFS)
    ds.bind("orcid", ORCID)
    ds.bind("prov", PROV)
    ds.bind("foaf", FOAF)
    ds.bind("sciencelive", SCIENCELIVE)
    ds.bind("hycl", HYCL)
    
    # HEAD
    head = ds.graph(head_graph)
    head.add((this_np, RDF.type, NP.Nanopublication))
    head.add((this_np, NP.hasAssertion, assertion_graph))
    head.add((this_np, NP.hasProvenance, provenance_graph))
    head.add((this_np, NP.hasPublicationInfo, pubinfo_graph))
    
    # ASSERTION
    assertion = ds.graph(assertion_graph)
    assertion.add((claim_uri, RDF.type, SCIENCELIVE["FORRT-Claim"]))
    assertion.add((claim_uri, RDF.type, forrt_type))
    assertion.add((claim_uri, RDFS.label, Literal(aida_sentence)))
    assertion.add((claim_uri, SCIENCELIVE["asAidaStatement"], aida_uri))
    
    # Source DOI (from statement or article)
    stmt_doi = statement.get("doi")
    if stmt_doi:
        doi_uri = URIRef(f"https://doi.org/{stmt_doi}" if not stmt_doi.startswith("http") else stmt_doi)
        assertion.add((claim_uri, DCT.source, doi_uri))
    elif article.get("dois"):
        assertion.add((claim_uri, DCT.source, URIRef(article["dois"])))
    
    # PROVENANCE
    provenance = ds.graph(provenance_graph)
    # Attribute to the original KL authors
    for auth in statement.get("authors", article.get("authors", [])):
        if auth.get("orcid"):
            provenance.add((assertion_graph, PROV.wasAttributedTo, URIRef(auth["orcid"])))
    # Also attribute to the nanopub creator
    provenance.add((assertion_graph, PROV.wasAttributedTo, author_uri))
    
    # PUBINFO
    pubinfo = ds.graph(pubinfo_graph)
    pubinfo.add((author_uri, FOAF.name, Literal(profile.name)))
    now = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%S.000Z")
    pubinfo.add((this_np, DCT.created, Literal(now, datatype=XSD.dateTime)))
    pubinfo.add((this_np, DCT.creator, author_uri))
    pubinfo.add((this_np, DCT.license, URIRef("https://creativecommons.org/licenses/by/4.0/")))
    pubinfo.add((this_np, NPX.wasCreatedAt, URIRef(PUBLISHER)))
    pubinfo.add((this_np, NPX.introduces, claim_uri))
    
    label = f"FORRT Claim: {aida_sentence[:80]}{'...' if len(aida_sentence) > 80 else ''}"
    pubinfo.add((this_np, RDFS.label, Literal(label)))
    pubinfo.add((this_np, NT.wasCreatedFromTemplate, FORRT_CLAIM_TEMPLATE))
    pubinfo.add((this_np, NT.wasCreatedFromProvenanceTemplate, PROV_TEMPLATE))
    pubinfo.add((this_np, NT.wasCreatedFromPubinfoTemplate, PUBINFO_TEMPLATE_1))
    pubinfo.add((this_np, NT.wasCreatedFromPubinfoTemplate, PUBINFO_TEMPLATE_2))
    
    # Add concept labels in pubinfo for Nanodash display
    for concept in statement.get("concepts", []):
        if concept.get("see_also"):
            pubinfo.add((URIRef(concept["see_also"]), NT.hasLabelFromApi, Literal(concept["label"])))
    
    return ds, label

print("create_forrt_claim() defined.")

In [9]:
def create_forrt_kl_study(record_name, analyses, paper_doi, script_url, gitlab_url, profile):
    """Create a FORRT-KL Replication Study nanopub from loom record data."""
    this_np = URIRef(TEMP_NP)
    head_graph = URIRef(TEMP_NP + "/Head")
    assertion_graph = URIRef(TEMP_NP + "/assertion")
    provenance_graph = URIRef(TEMP_NP + "/provenance")
    pubinfo_graph = URIRef(TEMP_NP + "/pubinfo")
    
    author_uri = ORCID[profile.orcid_id]
    study_uri = URIRef(TEMP_NP + f"/kl-study-{record_name}")
    
    ds = Dataset()
    ds.bind("this", TEMP_NP)
    ds.bind("sub", TEMP_NP)
    ds.bind("np", NP)
    ds.bind("dct", DCT)
    ds.bind("nt", NT)
    ds.bind("npx", NPX)
    ds.bind("xsd", XSD)
    ds.bind("rdfs", RDFS)
    ds.bind("orcid", ORCID)
    ds.bind("prov", PROV)
    ds.bind("foaf", FOAF)
    ds.bind("sciencelive", SCIENCELIVE)
    
    # HEAD
    head = ds.graph(head_graph)
    head.add((this_np, RDF.type, NP.Nanopublication))
    head.add((this_np, NP.hasAssertion, assertion_graph))
    head.add((this_np, NP.hasProvenance, provenance_graph))
    head.add((this_np, NP.hasPublicationInfo, pubinfo_graph))
    
    # ASSERTION
    assertion = ds.graph(assertion_graph)
    assertion.add((study_uri, RDF.type, SCIENCELIVE["FORRT-Replication-Study"]))
    assertion.add((study_uri, RDF.type, SCIENCELIVE["Reproduction-Study"]))
    
    label = f"Replication study of {record_name}"
    if paper_doi:
        label += f" ({paper_doi})"
    assertion.add((study_uri, RDFS.label, Literal(label)))
    
    assertion.add((study_uri, SCIENCELIVE["hasScopeDescription"],
                   Literal(f"Reproduction of analyses from Knowledge Loom record {record_name}")))
    assertion.add((study_uri, SCIENCELIVE["hasMethodologyDescription"],
                   Literal("Analyses reproduced using dtreg-generated machine-readable descriptions from the Knowledge Loom record.")))
    
    # Aggregate software info
    methods = set()
    packages = set()
    runtimes = set()
    input_descs = set()
    input_urls = set()
    
    for a in analyses:
        if a.method_call:
            methods.add(a.method_call.split("\n")[0].strip())
        elif a.method_label and a.package_name:
            methods.add(f"{a.package_name}::{a.method_label}()")
        if a.package_name:
            packages.add(f"{a.package_name} {a.package_version}".strip())
        if a.runtime_name:
            runtimes.add(f"{a.runtime_name} {a.runtime_version}".strip())
        if a.input_label:
            desc = a.input_label
            if a.input_rows and a.input_cols:
                desc += f" ({a.input_rows} rows x {a.input_cols} columns)"
            input_descs.add(desc)
        if a.input_source_url:
            input_urls.add(a.input_source_url)
    
    if methods:
        assertion.add((study_uri, SCIENCELIVE["executesMethod"], Literal("; ".join(sorted(methods)))))
    if packages:
        assertion.add((study_uri, SCIENCELIVE["usesSoftwarePackage"], Literal("; ".join(sorted(packages)))))
    if runtimes:
        assertion.add((study_uri, SCIENCELIVE["hasRuntimeEnvironment"], Literal("; ".join(sorted(runtimes)))))
    if input_descs:
        assertion.add((study_uri, SCIENCELIVE["hasInputDataDescription"], Literal("; ".join(sorted(input_descs)))))
    for url in sorted(input_urls):
        assertion.add((study_uri, SCIENCELIVE["hasInputDataSource"], URIRef(url)))
    if script_url:
        assertion.add((study_uri, SCIENCELIVE["hasAnalysisScript"], URIRef(script_url)))
    if gitlab_url:
        assertion.add((study_uri, SCIENCELIVE["hasLoomRecord"], URIRef(gitlab_url)))
    
    # PROVENANCE
    provenance = ds.graph(provenance_graph)
    provenance.add((assertion_graph, PROV.wasAttributedTo, author_uri))
    
    # PUBINFO
    pubinfo = ds.graph(pubinfo_graph)
    pubinfo.add((author_uri, FOAF.name, Literal(profile.name)))
    now = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%S.000Z")
    pubinfo.add((this_np, DCT.created, Literal(now, datatype=XSD.dateTime)))
    pubinfo.add((this_np, DCT.creator, author_uri))
    pubinfo.add((this_np, DCT.license, URIRef("https://creativecommons.org/licenses/by/4.0/")))
    pubinfo.add((this_np, NPX.wasCreatedAt, URIRef(PUBLISHER)))
    pubinfo.add((this_np, NPX.introduces, study_uri))
    pubinfo.add((this_np, RDFS.label, Literal(label)))
    pubinfo.add((this_np, NT.wasCreatedFromTemplate, FORRT_KL_STUDY_TEMPLATE))
    pubinfo.add((this_np, NT.wasCreatedFromProvenanceTemplate, PROV_TEMPLATE))
    pubinfo.add((this_np, NT.wasCreatedFromPubinfoTemplate, PUBINFO_TEMPLATE_1))
    pubinfo.add((this_np, NT.wasCreatedFromPubinfoTemplate, PUBINFO_TEMPLATE_2))
    
    return ds, label

print("create_forrt_kl_study() defined.")

create_forrt_kl_study() defined.


## Create FORRT-KL Replication Outcome nanopubs

In [10]:
def create_forrt_kl_outcome(record_name, analysis, index, gitlab_url, profile):
    """Create a FORRT-KL Replication Outcome nanopub for one analysis step."""
    this_np = URIRef(TEMP_NP)
    head_graph = URIRef(TEMP_NP + "/Head")
    assertion_graph = URIRef(TEMP_NP + "/assertion")
    provenance_graph = URIRef(TEMP_NP + "/provenance")
    pubinfo_graph = URIRef(TEMP_NP + "/pubinfo")
    
    author_uri = ORCID[profile.orcid_id]
    outcome_uri = URIRef(TEMP_NP + f"/kl-outcome-{record_name}-{index}")
    
    ds = Dataset()
    ds.bind("this", TEMP_NP)
    ds.bind("sub", TEMP_NP)
    ds.bind("np", NP)
    ds.bind("dct", DCT)
    ds.bind("nt", NT)
    ds.bind("npx", NPX)
    ds.bind("xsd", XSD)
    ds.bind("rdfs", RDFS)
    ds.bind("orcid", ORCID)
    ds.bind("prov", PROV)
    ds.bind("foaf", FOAF)
    ds.bind("schema", SCHEMA)
    ds.bind("sciencelive", SCIENCELIVE)
    
    method_str = f"{analysis.package_name}::{analysis.method_label}" if analysis.package_name else analysis.method_label
    label = f"Outcome: {method_str} from {record_name}"
    today = datetime.now(timezone.utc).strftime("%Y-%m-%d")
    
    # HEAD
    head = ds.graph(head_graph)
    head.add((this_np, RDF.type, NP.Nanopublication))
    head.add((this_np, NP.hasAssertion, assertion_graph))
    head.add((this_np, NP.hasProvenance, provenance_graph))
    head.add((this_np, NP.hasPublicationInfo, pubinfo_graph))
    
    # ASSERTION
    assertion = ds.graph(assertion_graph)
    assertion.add((outcome_uri, RDF.type, SCIENCELIVE["FORRT-Replication-Outcome"]))
    assertion.add((outcome_uri, RDFS.label, Literal(label)))
    assertion.add((outcome_uri, SCIENCELIVE["hasOutcomeRepository"], URIRef(gitlab_url)))
    assertion.add((outcome_uri, SCHEMA.endDate, Literal(today, datatype=XSD.date)))
    assertion.add((outcome_uri, SCIENCELIVE["hasValidationStatus"], SCIENCELIVE["Validated"]))
    assertion.add((outcome_uri, SCIENCELIVE["hasConfidenceLevel"], SCIENCELIVE["HighConfidence"]))
    assertion.add((outcome_uri, SCIENCELIVE["hasConclusionDescription"],
                   Literal(f"Analysis reproduced from Knowledge Loom record {record_name} using {method_str}.")))
    assertion.add((outcome_uri, SCIENCELIVE["hasEvidenceDescription"],
                   Literal("Machine-readable analysis proof available as dtreg JSON-LD in the Knowledge Loom record.")))
    
    # KL fields
    if analysis.json_file:
        proof_url = f"{gitlab_url}/-/raw/main/{analysis.json_file}"
        assertion.add((outcome_uri, SCIENCELIVE["hasMachineReadableProof"], URIRef(proof_url)))
    
    if analysis.analysis_type_hash and analysis.analysis_type_hash in DTREG_TYPE_MAP:
        assertion.add((outcome_uri, SCIENCELIVE["hasAnalysisType"], DTREG_TYPE_MAP[analysis.analysis_type_hash]))
    
    if analysis.output_rows and analysis.output_cols:
        assertion.add((outcome_uri, SCIENCELIVE["hasKeyResult"],
                       Literal(f"Output: {analysis.output_rows} rows x {analysis.output_cols} columns")))
    
    # PROVENANCE
    provenance = ds.graph(provenance_graph)
    provenance.add((assertion_graph, PROV.wasAttributedTo, author_uri))
    
    # PUBINFO
    pubinfo = ds.graph(pubinfo_graph)
    pubinfo.add((author_uri, FOAF.name, Literal(profile.name)))
    now = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%S.000Z")
    pubinfo.add((this_np, DCT.created, Literal(now, datatype=XSD.dateTime)))
    pubinfo.add((this_np, DCT.creator, author_uri))
    pubinfo.add((this_np, DCT.license, URIRef("https://creativecommons.org/licenses/by/4.0/")))
    pubinfo.add((this_np, NPX.wasCreatedAt, URIRef(PUBLISHER)))
    pubinfo.add((this_np, NPX.introduces, outcome_uri))
    pubinfo.add((this_np, RDFS.label, Literal(label)))
    pubinfo.add((this_np, NT.wasCreatedFromTemplate, FORRT_KL_OUTCOME_TEMPLATE))
    pubinfo.add((this_np, NT.wasCreatedFromProvenanceTemplate, PROV_TEMPLATE))
    pubinfo.add((this_np, NT.wasCreatedFromPubinfoTemplate, PUBINFO_TEMPLATE_1))
    pubinfo.add((this_np, NT.wasCreatedFromPubinfoTemplate, PUBINFO_TEMPLATE_2))
    
    return ds, label

print("create_forrt_kl_outcome() defined.")

create_forrt_kl_outcome() defined.


## Generate all nanopublications

In [ ]:
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
generated_files = []
record_name = LOOM_RESOURCE_ID

# 1. FORRT Claims (one per Knowledge Loom statement)
print("=== FORRT Claims ===")
for i, stmt in enumerate(statements):
    ds, label = create_forrt_claim(stmt, article, LOOM_RESOURCE_ID, profile)
    claim_file = Path(OUTPUT_DIR) / f"{record_name}-claim-{i+1}.trig"
    ds.serialize(destination=str(claim_file), format='trig')
    generated_files.append(claim_file)
    print(f"  {claim_file.name} — {label}")

# 2. FORRT-KL Replication Study (one per record, only if GitLab data available)
if analyses:
    print("\n=== FORRT-KL Replication Study ===")
    ds, label = create_forrt_kl_study(record_name, analyses, paper_doi, script_url, gitlab_url, profile)
    study_file = Path(OUTPUT_DIR) / f"{record_name}-study.trig"
    ds.serialize(destination=str(study_file), format='trig')
    generated_files.append(study_file)
    print(f"  {study_file.name} — {label}")

    # 3. FORRT-KL Replication Outcomes (one per analysis step)
    print("\n=== FORRT-KL Replication Outcomes ===")
    for i, analysis in enumerate(analyses):
        ds, label = create_forrt_kl_outcome(record_name, analysis, i+1, gitlab_url, profile)
        outcome_file = Path(OUTPUT_DIR) / f"{record_name}-outcome-{i+1}.trig"
        ds.serialize(destination=str(outcome_file), format='trig')
        generated_files.append(outcome_file)
        print(f"  {outcome_file.name} — {label}")
else:
    print("\n(No GitLab repo specified — skipping Study and Outcome nanopubs)")

print(f"\n{'='*40}")
print(f"Total: {len(generated_files)} nanopublications generated in {OUTPUT_DIR}")

## Preview

In [ ]:
# Preview the study nanopub
if generated_files:
    print(f"Preview of {generated_files[0]}:\n")
    print("=" * 80)
    with open(generated_files[0], 'r') as f:
        print(f.read())

## Sign and publish

In [ ]:
if PUBLISH:
    conf = NanopubConf(profile=profile)
    
    for trig_file in generated_files:
        np_obj = Nanopub(rdf=trig_file, conf=conf)
        np_obj.sign()
        
        signed_path = trig_file.with_suffix('.signed.trig')
        np_obj.store(signed_path)
        print(f"Signed: {signed_path}")
        
        np_obj.publish()
        print(f"Published: {np_obj.source_uri}")
else:
    print("Publishing disabled. Set PUBLISH = True to publish.")
    print(f"\nGenerated {len(generated_files)} files in {OUTPUT_DIR}")